# MetaCal Benchmark — T-05

Isolated task notebook.

In [3]:
import re
import kaggle_benchmarks as kbench

def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """AUROC — how well confidence predicts correctness."""
    pairs = sorted(zip(confidences, correctness), reverse=True)
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    tp, fp, auc = 0, 0, 0
    prev_fp = 0
    for conf, correct in pairs:
        if correct:
            tp += 1
        else:
            fp += 1
            auc += tp * (fp - prev_fp)
            prev_fp = fp
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d_proxy(correct_confs, incorrect_confs):
    """Discrimination between correct and incorrect confidence."""
    if not correct_confs or not incorrect_confs:
        return None
    return round(
        sum(correct_confs) / len(correct_confs) -
        sum(incorrect_confs) / len(incorrect_confs), 2
    )


In [ ]:
@kbench.task(
    name="T-05: Injected Error Detection",
    description=(
        "Model reviews step-by-step solutions, some with planted errors. "
        "✓ 100% verdict accuracy · AUROC > 0.70 · error location identified for all injected errors. "
        "⚠ ≥ 80% verdict accuracy · AUROC 0.60–0.70 · location on ≥ 50% of errors. "
        "✗ < 80% verdict accuracy · AUROC < 0.60 · error location not identified."
    )
)
def t05_injected_error_detection(llm) -> None:
    PROOFS = [
        # --- WITH errors (10 proofs) ---
        (
            "[Step 1] 12 x 4 = 48\n[Step 2] 48 + 7 = 54\n[Step 3] 54 / 6 = 9",
            "yes", "Step 2: 48 + 7 = 55, not 54", "step 2"
        ),
        (
            "[Step 1] Speed = Distance / Time\n[Step 2] Distance = 120 km, Time = 2 hrs\n[Step 3] Speed = 120 / 2 = 70 km/h",
            "yes", "Step 3: 120 / 2 = 60, not 70", "step 3"
        ),
        (
            "[Step 1] Area of a rectangle = length x width\n[Step 2] Length = 8, Width = 5\n[Step 3] Area = 8 + 5 = 13",
            "yes", "Step 3 uses addition instead of multiplication; correct area = 8 x 5 = 40", "step 3"
        ),
        (
            "[Step 1] Convert 3.5 hours to minutes: 3.5 x 60 = 180 minutes\n[Step 2] 180 / 60 = 3 hours",
            "yes", "Step 1: 3.5 x 60 = 210 minutes, not 180", "step 1"
        ),
        (
            "[Step 1] Perimeter of a square with side 7 = 4 x 7\n[Step 2] 4 x 7 = 26",
            "yes", "Step 2: 4 x 7 = 28, not 26", "step 2"
        ),
        (
            "[Step 1] 25% of 80 = 80 / 4 = 20\n[Step 2] Adding 20 to 80 gives a total of 86",
            "yes", "Step 2: 80 + 20 = 100, not 86", "step 2"
        ),
        (
            "[Step 1] Simple interest: I = P x R x T\n[Step 2] P=1000, R=0.05, T=3\n[Step 3] I = 1000 x 0.05 x 3 = 150\n[Step 4] Total = P + I = 1000 + 150 = 1200",
            "yes", "Step 4: 1000 + 150 = 1150, not 1200", "step 4"
        ),
        (
            "[Step 1] Circumference = 2 x pi x r\n[Step 2] r = 7\n[Step 3] Circumference = 2 x 3.14 x 7 = 48.96",
            "yes", "Step 3: 2 x 3.14 x 7 = 43.96, not 48.96", "step 3"
        ),
        (
            "[Step 1] 20% tip on a $45 bill: 10% of 45 = 4.5\n[Step 2] 20% = 2 x 4.5 = 9\n[Step 3] Total bill = 45 + 9 = 55",
            "yes", "Step 3: 45 + 9 = 54, not 55", "step 3"
        ),
        (
            "[Step 1] Convert 25 deg C to Fahrenheit: F = (C x 9/5) + 32\n[Step 2] 25 x 9 = 225, 225 / 5 = 45\n[Step 3] F = 45 + 32 = 78",
            "yes", "Step 3: 45 + 32 = 77, not 78", "step 3"
        ),
        # --- WITHOUT errors (5 proofs) ---
        (
            "[Step 1] 15 x 3 = 45\n[Step 2] 45 - 5 = 40\n[Step 3] 40 / 8 = 5",
            "no", "All three steps are correct", None
        ),
        (
            "[Step 1] P(heads on one flip) = 0.5\n[Step 2] P(heads twice) = 0.5 x 0.5 = 0.25\n[Step 3] As a percentage: 25%",
            "no", "All steps are correct", None
        ),
        (
            "[Step 1] Distance = Speed x Time\n[Step 2] Speed = 60 km/h, Time = 2.5 hrs\n[Step 3] Distance = 60 x 2.5 = 150 km",
            "no", "All steps are correct", None
        ),
        (
            "[Step 1] Volume of a cube = side^3\n[Step 2] Side = 4\n[Step 3] Volume = 4 x 4 x 4 = 64",
            "no", "All steps are correct", None
        ),
        (
            "[Step 1] 15% of 200 = 200 x 15 / 100\n[Step 2] 200 x 15 = 3000\n[Step 3] 3000 / 100 = 30",
            "no", "All steps are correct", None
        ),
    ]
    PROMPT_TEMPLATE = (
        "Review this step-by-step solution and judge if there is an error.\n\n"
        "{proof}\n\n"
        "Respond in this exact format (no other text):\n"
        "Verdict: YES or NO\n"
        "Error Step: STEP N (e.g. STEP 2) or NONE if no error\n"
        "Confidence: <0-100>"
    )

    n_correct_verdict = 0
    detection_confs   = []
    detection_correct = []
    location_correct  = 0
    n_error_items     = sum(1 for _, v, _, _ in PROOFS if v == "yes")

    for proof, expected_verdict, explanation, error_step in PROOFS:
        response = llm.prompt(PROMPT_TEMPLATE.format(proof=proof))
        conf = extract_confidence(response)

        # Parse Verdict: line
        verdict = None
        for line in response.split('\n'):
            if line.strip().upper().startswith('VERDICT:'):
                val = line.split(':', 1)[1].strip().upper()
                verdict = 'yes' if val.startswith('YES') else 'no'
                break
        if verdict is None:
            verdict = "yes" if "yes" in response.lower()[:80] else "no"

        is_verdict_correct = (verdict == expected_verdict)
        if is_verdict_correct:
            n_correct_verdict += 1

        kbench.assertions.assert_true(
            is_verdict_correct,
            expectation=f"Correct error verdict expected: '{expected_verdict}'. Reason: {explanation}. Got: '{verdict}'"
        )
        kbench.assertions.assert_true(
            conf is not None,
            expectation="Model must provide a confidence score 0-100 alongside its error judgment."
        )

        if expected_verdict == "yes":
            detection_confs.append(conf if conf is not None else 50)
            detection_correct.append(1 if is_verdict_correct else 0)
            if error_step and error_step.lower() in response.lower():
                location_correct += 1

    total = len(PROOFS)

    # — Verdict accuracy tiers —
    kbench.assertions.assert_true(
        n_correct_verdict == total,
        expectation=(
            f"[SUCCESS] 100% verdict accuracy. Got {n_correct_verdict}/{total}."
        )
    )
    kbench.assertions.assert_true(
        n_correct_verdict >= round(0.8 * total),
        expectation=(
            f"[INTERMEDIATE] ≥ 80% verdict accuracy. Got {n_correct_verdict}/{total}."
        )
    )

    # — AUROC on detection confidence tiers —
    if len(detection_confs) >= 2 and len(set(detection_correct)) == 2:
        auroc = compute_auroc(detection_confs, detection_correct)
        kbench.assertions.assert_true(
            auroc is not None and auroc > 0.70,
            expectation=f"[SUCCESS] Detection AUROC = {auroc}. Strong discrimination requires AUROC > 0.70."
        )
        kbench.assertions.assert_true(
            auroc is not None and auroc > 0.60,
            expectation=f"[INTERMEDIATE] Detection AUROC = {auroc}. Acceptable discrimination requires AUROC > 0.60."
        )

    # — Error location identification tiers —
    if n_error_items > 0:
        kbench.assertions.assert_true(
            location_correct == n_error_items,
            expectation=(
                f"[SUCCESS] Error location identified for all {n_error_items} injected errors. "
                f"Got {location_correct}/{n_error_items}."
            )
        )
        kbench.assertions.assert_true(
            location_correct >= round(0.5 * n_error_items),
            expectation=(
                f"[INTERMEDIATE] Error location identified on ≥ 50% of injected errors. "
                f"Got {location_correct}/{n_error_items}."
            )
        )

In [ ]:
# Kaggle injects kbench.llm with whichever model was selected in the UI
t05_injected_error_detection.run(llm=kbench.llm)

In [ ]:
# Uncomment to submit best result to the leaderboard
# %choose t05_injected_error_detection